In [1]:
import os
from dotenv import load_dotenv


PROJECT_DIR = os.environ["PROJECT_DIR"]
load_dotenv(f"{PROJECT_DIR}/memo/.env")
MODEL_ID="jp.anthropic.claude-opus-4-8"

In [2]:
#  asyncio は「イベントループは入れ子（ネスト）にできない」という制約を回避するためのコード
import nest_asyncio
nest_asyncio.apply()

In [3]:
from dataclasses import dataclass

@dataclass
class InventoryItem:
    name: str
    quantity_on_hand: int
    weekly_quantity_sold_past_n_weeks: list[int]
    weeks_to_deliver: int

@dataclass
class Reorder:
    name: str
    quantity_to_order: int
    reason_to_reorder: str

items = [
    InventoryItem("itemA", 300, [50, 70, 80, 100], 2),
    InventoryItem("itemB", 100, [70, 80, 90, 70], 2),
    InventoryItem("itemC", 200, [80, 70, 90, 80], 1)
]

In [4]:
# https://pydantic.dev/docs/ai/models/bedrock/

from pydantic_ai import Agent

agent = Agent(
    f"bedrock:{MODEL_ID}",
    system_prompt="あなたは、必要な時に必要なだけ発注する在庫管理者です。",
    output_type=list[Reorder]
)

result = agent.run_sync(f"""
これらの品目のうち、今週中に再注文する必要があるものを特定してください。

**Items**
{items}
""")
print(result.output)

[Reorder(name='itemB', quantity_to_order=240, reason_to_reorder='週平均販売数は約77.5個（直近のトレンドはやや減少傾向）。納期は2週間で、その間に約160個（80×2）が販売される見込みだが、現在の在庫は100個しかなく不足する。納期＋安全在庫1週間分を考慮すると約240個（3週間分）の発注が必要。今週中に再注文すべき。'), Reorder(name='itemC', quantity_to_order=80, reason_to_reorder='週平均販売数は約80個。納期は1週間で、現在の在庫200個は約2.5週間分に相当するため当面の不足リスクは低いが、安全在庫を維持するため納期1週間分＋αとして約80個を補充発注しておくのが望ましい。')]
